# CT-CLIP Grad-CAM — standalone, single-cell run

Predicts on a case pool, picks the best nodule + best GGO cases separately, runs Grad-CAM, scores against ground truth, saves everything to disk, and shows every triptych inline.

In [1]:
import sys
from pathlib import Path

SCRIPTS_DIR = Path("/home/chest_ct/code/models/ct-clip/scripts")
sys.path.insert(0, str(SCRIPTS_DIR))

import numpy as np
import pandas as pd
import nibabel as nib
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from rexground_pilot_pipeline import (
    load_ctclip_classifier, preprocess_ct, run_gradcam,
    VOLUME_DIR, SEG_DIR, TRAIN_META, VALID_META,
    LUNG_NODULE_IDX, LUNG_OPACITY_IDX, DEVICE, ROOT,
)
from rexground_full_pipeline import build_full_case_list, run_predictions, gradcam_overlap
from visualize_gradcam_top20 import aligned_gt_mask, best_dice_slice

print("device:", DEVICE)

N_CASES = 30      # how many cases (by lesion size) to run predictions on; None = all ~2000
N_BEST = 5        # how many best cases PER pathology to run Grad-CAM + visualize (5 nodule + 5 ggo = 10 total)

OUT_DIR = ROOT / "rexground_predictions/standalone_run"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CAM_DIR = OUT_DIR / "cams"
CAM_DIR.mkdir(parents=True, exist_ok=True)
print("output dir:", OUT_DIR)

cases = build_full_case_list()
n_nodule = sum(1 for c in cases if c["kind"] == "nodule")
n_ggo = sum(1 for c in cases if c["kind"] == "ggo")
n_mixed = sum(1 for c in cases if c["kind"] == "mixed")
print(f"{len(cases)} total cases  ({n_nodule} nodule-only, {n_ggo} ggo-only, {n_mixed} mixed)")

model, tokenizer = load_ctclip_classifier()
train_meta = pd.read_csv(TRAIN_META).set_index("VolumeName")
valid_meta = pd.read_csv(VALID_META).set_index("VolumeName")

cases_by_lesion_size = sorted(cases, key=lambda c: -sum(c["case"].get("pixels", {}).values()))
case_subset = cases if N_CASES is None else cases_by_lesion_size[:N_CASES]

predictions_df = run_predictions(model, tokenizer, case_subset, train_meta, valid_meta)
predictions_df.to_csv(OUT_DIR / "predictions.csv", index=False)
print("saved:", OUT_DIR / "predictions.csv")
predictions_df.head()

valid_preds = predictions_df[predictions_df["error"].isna()] if "error" in predictions_df.columns else predictions_df

nodule_pool = valid_preds[valid_preds["kind"].isin(["nodule", "mixed"])].copy()
nodule_pool["gradcam_target"] = "nodule"
nodule_pool["target_logit"] = nodule_pool["logit_lung_nodule"]

ggo_pool = valid_preds[valid_preds["kind"].isin(["ggo", "mixed"])].copy()
ggo_pool["gradcam_target"] = "ggo"
ggo_pool["target_logit"] = ggo_pool["logit_lung_opacity"]

top_nodule = nodule_pool.sort_values("target_logit", ascending=False).head(N_BEST)
top_ggo = ggo_pool.sort_values("target_logit", ascending=False).head(N_BEST)
top_cases = pd.concat([top_nodule, top_ggo]).to_dict("records")

pd.DataFrame(top_cases)[["name", "kind", "gradcam_target", "target_logit", "logit_lung_nodule", "logit_lung_opacity"]]

gradcam_results = []

for rec in top_cases:
    name = rec["name"]
    target = rec["gradcam_target"]
    target_idx = LUNG_NODULE_IDX if target == "nodule" else LUNG_OPACITY_IDX
    meta_row = train_meta.loc[name] if name in train_meta.index else valid_meta.loc[name]

    image_tensor, crop_info = preprocess_ct(VOLUME_DIR / name, meta_row)
    cam, probs = run_gradcam(model, tokenizer, image_tensor, target_idx)

    seg_nii = nib.load(str(SEG_DIR / name))
    seg = seg_nii.get_fdata()
    if seg.ndim == 4:
        seg = seg[0]
    seg_mask = seg > 0

    best_dice, best_pct, pointing_hit = gradcam_overlap(cam, seg_mask, crop_info, meta_row)
    target_prob = float(probs[target_idx])

    cam_path = CAM_DIR / f"gradcam_{target}_{name.replace('.nii.gz', '')}.npy"
    np.save(cam_path, cam)

    print(f"{target:6s} {name:28s} prob={target_prob:.3f} "
          f"best_dice@{best_pct}%={best_dice:.4f} pointing_hit={pointing_hit}")

    gradcam_results.append({
        **rec, "target_prob": target_prob, "cam": cam, "crop_info": crop_info,
        "meta_row": meta_row, "gradcam_best_dice": best_dice,
        "gradcam_best_pct": best_pct, "gradcam_pointing_hit": pointing_hit,
        "gradcam_npy": str(cam_path),
    })

summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ("cam", "crop_info", "meta_row", "case")}
    for r in gradcam_results
])
summary_df.to_csv(OUT_DIR / "gradcam_summary.csv", index=False)

print(f"n cases          : {len(summary_df)}")
print(f"avg best dice     : {summary_df['gradcam_best_dice'].mean():.4f}")
print(f"pointing accuracy : {summary_df['gradcam_pointing_hit'].mean()*100:.2f}%")
summary_df[["name", "gradcam_target", "target_prob", "gradcam_best_dice", "gradcam_best_pct", "gradcam_pointing_hit"]]

for r in gradcam_results:
    name = r["name"]
    target = r["gradcam_target"]
    cam = r["cam"]
    crop_info = r["crop_info"]
    meta_row = r["meta_row"]

    image_tensor, _ = preprocess_ct(VOLUME_DIR / name, meta_row)
    img_zxy = image_tensor.squeeze().numpy()
    img_xyz = np.transpose(img_zxy, (1, 2, 0))          # X,Y,Z

    cam_xyz = np.transpose(cam, (1, 2, 0))              # X,Y,Z
    cam_norm = cam_xyz - cam_xyz.min()
    cam_norm = cam_norm / (cam_norm.max() + 1e-8)

    gt_mask = aligned_gt_mask(name, crop_info, meta_row)  # X,Y,Z
    z, slice_dice = best_dice_slice(cam_xyz, gt_mask, r["gradcam_best_pct"])

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(img_xyz[:, :, z], cmap="gray")
    axes[0].set_title(f"{name}\nCT slice {z}")
    axes[0].axis("off")

    axes[1].imshow(img_xyz[:, :, z], cmap="gray")
    axes[1].imshow(
        np.ma.masked_where(gt_mask[:, :, z] == 0, gt_mask[:, :, z]),
        cmap="Reds", alpha=0.6,
    )
    axes[1].set_title("Ground truth (segmentation)")
    axes[1].axis("off")

    axes[2].imshow(img_xyz[:, :, z], cmap="gray")
    axes[2].imshow(cam_norm[:, :, z], cmap="jet", alpha=0.5, vmin=0, vmax=1)
    axes[2].contour(gt_mask[:, :, z], levels=[0.5], colors="white", linewidths=1)
    axes[2].set_title(
        f"CT-CLIP Grad-CAM ({target})\nslice Dice={slice_dice:.4f}  |  case-best Dice={r['gradcam_best_dice']:.4f}"
    )
    axes[2].axis("off")

    plt.tight_layout()

    viz_dir = OUT_DIR / "visualizations"
    viz_dir.mkdir(parents=True, exist_ok=True)
    fig_path = viz_dir / f"{target}_{name.replace('.nii.gz', '')}.png"
    plt.savefig(fig_path, dpi=110)
    print("saved viz:", fig_path)
    plt.show()
    plt.close(fig)


device: cuda
output dir: /home/chest_ct/code/models/ct-clip/rexground_predictions/standalone_run
2036 total cases  (1104 nodule-only, 703 ggo-only, 229 mixed)


test all pooling


test all pooling


test all pooling


test all pooling


...

saved viz: /home/chest_ct/code/models/ct-clip/rexground_predictions/standalone_run/visualizations/ggo_train_655_a_2.png


saved viz: /home/chest_ct/code/models/ct-clip/rexground_predictions/standalone_run/visualizations/ggo_train_1593_b_2.png


saved viz: /home/chest_ct/code/models/ct-clip/rexground_predictions/standalone_run/visualizations/ggo_train_2333_a_2.png


saved viz: /home/chest_ct/code/models/ct-clip/rexground_predictions/standalone_run/visualizations/ggo_train_6746_b_1.png


saved viz: /home/chest_ct/code/models/ct-clip/rexground_predictions/standalone_run/visualizations/ggo_train_2276_a_2.png
